# HotelReservationAction

Execute reservation create/change/cancel actions with typed parameters, explicit confirmation IDs,
idempotency keys, optimistic concurrency, Delta MERGE, and immutable audit events.

In [ ]:
action = "create"
confirmation_id = "00000000-0000-0000-0000-000000000000"
idempotency_key = "00000000-0000-0000-0000-000000000000"
expected_record_version = 1
request_timestamp_utc = "2026-07-11T19:00:00Z"
request_payload_json = "{}"
workspace_id = "00000000-0000-0000-0000-000000000000"
hotel_guest_services_lakehouse_id = "00000000-0000-0000-0000-000000000000"
evaluation_snapshot_id = ""

In [ ]:
from __future__ import annotations

import json
import hashlib
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

import notebookutils
from delta.tables import DeltaTable
from pyspark.errors import AnalysisException
from pyspark.sql import DataFrame, functions as F, types as T

DATE_START = date(2026, 3, 1)
DATE_END = date(2027, 3, 31)

SOURCE_TABLE_SPECS: dict[str, dict[str, object]] = {
    "properties": {
        "lakehouse": "operations",
        "primary_key": ["property_id"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
            T.StructField("region", T.StringType(), False),
            T.StructField("city", T.StringType(), False),
            T.StructField("country", T.StringType(), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("tax_rate", T.DecimalType(10, 4), False),
            T.StructField("facilities", T.StringType(), False),
            T.StructField("pool_hours", T.StringType(), False),
            T.StructField("parking", T.StringType(), False),
            T.StructField("transport", T.StringType(), False),
        ]),
    },
    "room_types": {
        "lakehouse": "operations",
        "primary_key": ["room_type_id"],
        "schema": T.StructType([
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
            T.StructField("capacity", T.IntegerType(), False),
            T.StructField("accessible", T.BooleanType(), False),
            T.StructField("base_rate", T.DecimalType(10, 2), False),
        ]),
    },
    "rooms": {
        "lakehouse": "operations",
        "primary_key": ["room_id"],
        "schema": T.StructType([
            T.StructField("room_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("room_number", T.StringType(), False),
            T.StructField("room_status", T.StringType(), False),
        ]),
    },
    "amenities": {
        "lakehouse": "operations",
        "primary_key": ["amenity_id"],
        "schema": T.StructType([
            T.StructField("amenity_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
        ]),
    },
    "property_amenities": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "amenity_id"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("amenity_id", T.StringType(), False),
        ]),
    },
    "daily_inventory": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "room_type_id", "service_date"],
        "date_columns": ["service_date"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("service_date", T.DateType(), False),
            T.StructField("physical_rooms", T.IntegerType(), False),
            T.StructField("out_of_order_rooms", T.IntegerType(), False),
            T.StructField("sellable_rooms", T.IntegerType(), False),
            T.StructField("occupied_rooms", T.IntegerType(), False),
        ]),
    },
    "daily_rates": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "room_type_id", "service_date", "rate_plan"],
        "date_columns": ["service_date"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("service_date", T.DateType(), False),
            T.StructField("rate_plan", T.StringType(), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("rate", T.DecimalType(10, 2), False),
        ]),
    },
    "property_daily_metrics": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "service_date"],
        "date_columns": ["service_date"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("service_date", T.DateType(), False),
            T.StructField("sellable_rooms", T.IntegerType(), False),
            T.StructField("occupied_rooms", T.IntegerType(), False),
            T.StructField("room_revenue", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
        ]),
    },
    "outlets": {
        "lakehouse": "operations",
        "primary_key": ["outlet_id"],
        "schema": T.StructType([
            T.StructField("outlet_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
            T.StructField("outlet_type", T.StringType(), False),
            T.StructField("capacity", T.IntegerType(), False),
        ]),
    },
    "guests": {
        "lakehouse": "guest_services",
        "primary_key": ["guest_id"],
        "schema": T.StructType([
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("first_name", T.StringType(), False),
            T.StructField("last_name", T.StringType(), False),
            T.StructField("email", T.StringType(), False),
            T.StructField("country", T.StringType(), False),
            T.StructField("privacy_consent", T.BooleanType(), False),
        ]),
    },
    "reservations": {
        "lakehouse": "guest_services",
        "primary_key": ["reservation_id"],
        "date_columns": ["check_in", "check_out"],
        "schema": T.StructType([
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("check_in", T.DateType(), False),
            T.StructField("check_out", T.DateType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("cancellation_policy", T.StringType(), False),
            T.StructField("room_count", T.IntegerType(), False),
            T.StructField("rate_total", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("channel", T.StringType(), False),
        ]),
    },
    "stays": {
        "lakehouse": "guest_services",
        "primary_key": ["stay_id"],
        "schema": T.StructType([
            T.StructField("stay_id", T.StringType(), False),
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("room_id", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("actual_check_in", T.TimestampType(), False),
            T.StructField("actual_check_out", T.TimestampType(), True),
        ]),
    },
    "restaurant_reservations": {
        "lakehouse": "guest_services",
        "primary_key": ["restaurant_reservation_id"],
        "schema": T.StructType([
            T.StructField("restaurant_reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("outlet_id", T.StringType(), False),
            T.StructField("reservation_at", T.TimestampType(), False),
            T.StructField("party_size", T.IntegerType(), False),
            T.StructField("status", T.StringType(), False),
        ]),
    },
    "folios": {
        "lakehouse": "guest_services",
        "primary_key": ["folio_id"],
        "schema": T.StructType([
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("stay_id", T.StringType(), False),
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("total_charges", T.DecimalType(14, 2), False),
            T.StructField("total_adjustments", T.DecimalType(14, 2), False),
            T.StructField("total_payments", T.DecimalType(14, 2), False),
            T.StructField("total_refunds", T.DecimalType(14, 2), False),
            T.StructField("balance", T.DecimalType(14, 2), False),
        ]),
    },
    "folio_transactions": {
        "lakehouse": "guest_services",
        "primary_key": ["folio_transaction_id"],
        "schema": T.StructType([
            T.StructField("folio_transaction_id", T.StringType(), False),
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("transaction_type", T.StringType(), False),
            T.StructField("description", T.StringType(), False),
            T.StructField("amount", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("posted_at", T.TimestampType(), False),
        ]),
    },
    "payments": {
        "lakehouse": "guest_services",
        "primary_key": ["payment_id"],
        "schema": T.StructType([
            T.StructField("payment_id", T.StringType(), False),
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("payment_method", T.StringType(), False),
            T.StructField("payment_token", T.StringType(), False),
            T.StructField("card_mask", T.StringType(), False),
            T.StructField("amount", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("payment_status", T.StringType(), False),
        ]),
    },
    "billing_disputes": {
        "lakehouse": "guest_services",
        "primary_key": ["billing_dispute_id"],
        "schema": T.StructType([
            T.StructField("billing_dispute_id", T.StringType(), False),
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("dispute_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("amount", T.DecimalType(14, 2), False),
            T.StructField("opened_at", T.TimestampType(), False),
        ]),
    },
    "special_requests": {
        "lakehouse": "guest_services",
        "primary_key": ["special_request_id"],
        "schema": T.StructType([
            T.StructField("special_request_id", T.StringType(), False),
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("request_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("requested_at", T.TimestampType(), False),
            T.StructField("details", T.StringType(), False),
        ]),
    },
    "maintenance_requests": {
        "lakehouse": "operations",
        "primary_key": ["maintenance_request_id"],
        "schema": T.StructType([
            T.StructField("maintenance_request_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("stay_id", T.StringType(), False),
            T.StructField("room_id", T.StringType(), False),
            T.StructField("request_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("reported_at", T.TimestampType(), False),
        ]),
    },
    "work_orders": {
        "lakehouse": "operations",
        "primary_key": ["work_order_id"],
        "schema": T.StructType([
            T.StructField("work_order_id", T.StringType(), False),
            T.StructField("maintenance_request_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("priority", T.StringType(), False),
            T.StructField("opened_at", T.TimestampType(), False),
        ]),
    },
    "action_confirmations": {
        "lakehouse": "guest_services",
        "primary_key": ["confirmation_id"],
        "schema": T.StructType([
            T.StructField("confirmation_id", T.StringType(), False),
            T.StructField("action_type", T.StringType(), False),
            T.StructField("target_id", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("confirmed_at", T.TimestampType(), False),
        ]),
    },
    "idempotency_records": {
        "lakehouse": "guest_services",
        "primary_key": ["idempotency_key"],
        "schema": T.StructType([
            T.StructField("idempotency_key", T.StringType(), False),
            T.StructField("action_type", T.StringType(), False),
            T.StructField("result_id", T.StringType(), False),
            T.StructField("created_at", T.TimestampType(), False),
        ]),
    },
    "audit_events": {
        "lakehouse": "guest_services",
        "primary_key": ["audit_event_id"],
        "schema": T.StructType([
            T.StructField("audit_event_id", T.StringType(), False),
            T.StructField("entity_type", T.StringType(), False),
            T.StructField("entity_id", T.StringType(), False),
            T.StructField("action", T.StringType(), False),
            T.StructField("occurred_at", T.TimestampType(), False),
            T.StructField("actor_type", T.StringType(), False),
        ]),
    },
}

MUTABLE_TECHNICAL_COLUMNS = {
    "record_version": T.IntegerType(),
    "last_modified_at": T.TimestampType(),
    "last_confirmation_id": T.StringType(),
    "pending_reconciliation": T.BooleanType(),
    "is_fixture": T.BooleanType(),
}

MUTABLE_TABLES = {"reservations", "restaurant_reservations", "billing_disputes", "special_requests", "maintenance_requests", "work_orders"}

PROPERTY_CURRENCY_TABLES = {"reservations": "currency", "folios": "currency", "payments": "currency", "daily_rates": "currency", "property_daily_metrics": "currency"}

FIXTURE_KEYS = {
    "reservation": {"family_accessibility", "rate_change", "cancellable", "nonrefundable", "folio_dispute", "restaurant"},
    "request": {"prearrival_request", "maintenance"},
}


def table_path(workspace_id: str, lakehouse_id: str, table_name: str) -> str:
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{table_name}"


def files_path(workspace_id: str, lakehouse_id: str, relative_path: str) -> str:
    trimmed = relative_path.lstrip("/")
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/{trimmed}"


def resolve_local_or_abfss(source_root: str, child: str) -> str:
    if source_root.startswith("abfss://"):
        return source_root.rstrip("/") + "/" + child
    return str((Path(source_root) / child).resolve())


def read_source_table(source_root: str, table_name: str) -> DataFrame:
    spec = SOURCE_TABLE_SPECS[table_name]
    path = resolve_local_or_abfss(source_root, f"{table_name}.csv")
    return spark.read.option("header", True).schema(spec["schema"]).csv(path)


def read_json_document(source_root: str, child: str) -> dict[str, object]:
    path = resolve_local_or_abfss(source_root, child)
    if path.startswith("abfss://"):
        text = notebookutils.fs.head(path, 10_000_000)
    else:
        text = Path(path).read_text(encoding="utf-8")
    return json.loads(text)


def assert_no_duplicates(df: DataFrame, keys: list[str], label: str) -> None:
    duplicates = df.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count()
    if duplicates:
        raise ValueError(f"Duplicate keys detected in {label}: {keys}")


def assert_row_count(df: DataFrame, expected: int, label: str) -> None:
    actual = df.count()
    if actual != expected:
        raise ValueError(f"{label} row count mismatch: expected {expected}, found {actual}")


def assert_foreign_key(child: DataFrame, child_key: str, parent: DataFrame, parent_key: str, label: str) -> None:
    missing = child.select(child_key).distinct().join(parent.select(parent_key).distinct(), F.col(child_key) == F.col(parent_key), "left_anti").limit(1).count()
    if missing:
        raise ValueError(f"Foreign-key validation failed for {label}: {child_key} -> {parent_key}")


def assert_date_window(df: DataFrame, column_name: str, label: str) -> None:
    bounds = df.select(F.min(column_name).alias("min_date"), F.max(column_name).alias("max_date")).first()
    min_date = bounds["min_date"]
    max_date = bounds["max_date"]
    if min_date is None or max_date is None or min_date < DATE_START or max_date > DATE_END:
        raise ValueError(f"{label}.{column_name} is outside the supported date window {DATE_START}..{DATE_END}")


def assert_property_currency(df: DataFrame, currency_column: str, properties_df: DataFrame, label: str) -> None:
    invalid = (
        df.alias("fact")
        .join(properties_df.select("property_id", F.col("currency").alias("property_currency")).alias("dim"), "property_id", "inner")
        .where(F.col(currency_column) != F.col("property_currency"))
        .limit(1)
        .count()
    )
    if invalid:
        raise ValueError(f"Currency mismatch detected in {label}")


def add_mutation_metadata(df: DataFrame, confirmation_id: str | None = None, pending_reconciliation: bool = False, is_fixture: bool = True) -> DataFrame:
    result = df
    if "record_version" not in result.columns:
        result = result.withColumn("record_version", F.lit(1))
    if "last_modified_at" not in result.columns:
        result = result.withColumn("last_modified_at", F.current_timestamp())
    if "last_confirmation_id" not in result.columns:
        result = result.withColumn("last_confirmation_id", F.lit(confirmation_id).cast(T.StringType()))
    if "pending_reconciliation" not in result.columns:
        result = result.withColumn("pending_reconciliation", F.lit(pending_reconciliation))
    if "is_fixture" not in result.columns:
        result = result.withColumn("is_fixture", F.lit(is_fixture))
    return result


def overwrite_delta(df: DataFrame, workspace_id: str, lakehouse_id: str, table_name: str) -> None:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path(workspace_id, lakehouse_id, table_name)))


def read_delta(workspace_id: str, lakehouse_id: str, table_name: str) -> DataFrame:
    return spark.read.format("delta").load(table_path(workspace_id, lakehouse_id, table_name))


def merge_single_row(workspace_id: str, lakehouse_id: str, table_name: str, key_column: str, payload_df: DataFrame, update_assignments: dict[str, str], insert_assignments: dict[str, str]) -> None:
    delta = DeltaTable.forPath(spark, table_path(workspace_id, lakehouse_id, table_name))
    (delta.alias("target").merge(payload_df.alias("source"), f"target.{key_column} = source.{key_column}")
        .whenMatchedUpdate(set=update_assignments)
        .whenNotMatchedInsert(values=insert_assignments)
        .execute())


def request_hash(payload_json: str) -> str:
    return hashlib.sha256(payload_json.encode("utf-8")).hexdigest()

In [ ]:
def load_json_payload(payload_json: str) -> dict[str, object]:
    payload = json.loads(payload_json)
    if not isinstance(payload, dict):
        raise ValueError("request_payload_json must decode to a JSON object.")
    return payload


def require_evaluation_write_fence() -> None:
    lock = files_path(
        workspace_id,
        hotel_guest_services_lakehouse_id,
        "assert-evaluation-state/active.json",
    )
    if not notebookutils.fs.exists(lock):
        if evaluation_snapshot_id:
            raise RuntimeError("Evaluation write rejected because its state lock is missing.")
        return
    lock_value = json.loads(notebookutils.fs.head(lock, 1024 * 1024))
    if not isinstance(lock_value, dict):
        raise RuntimeError("Evaluation write rejected because the state lock is invalid.")
    if not evaluation_snapshot_id:
        raise RuntimeError("Reservation writes are blocked while an evaluation owns Fabric state.")
    if lock_value.get("snapshot_id") != evaluation_snapshot_id:
        raise RuntimeError("Evaluation write rejected because the state lock has a different owner.")
    if lock_value.get("status") != "active":
        raise RuntimeError("Evaluation write rejected because restoration has begun.")


def require_fields(payload: dict[str, object], field_names: list[str]) -> None:
    missing = [field_name for field_name in field_names if field_name not in payload or payload[field_name] in {None, ""}]
    if missing:
        raise ValueError(f"request_payload_json is missing required fields: {missing}")


def get_scalar(df: DataFrame, column_name: str):
    row = df.select(column_name).first()
    return None if row is None else row[column_name]


def ensure_idempotency(workspace_id: str, guest_services_lakehouse_id: str, action_type: str, payload_json: str) -> DataFrame | None:
    existing = read_delta(workspace_id, guest_services_lakehouse_id, "idempotency_records").where(F.col("idempotency_key") == idempotency_key)
    if existing.limit(1).count() == 0:
        return None
    existing_hash = get_scalar(existing, "request_hash")
    if existing_hash and existing_hash != request_hash(payload_json):
        raise RuntimeError("Conflict: idempotency key was already used for a different payload.")
    return existing


def merge_control_record(table_name: str, key_column: str, row: dict[str, object]) -> None:
    schema = read_delta(workspace_id, hotel_guest_services_lakehouse_id, table_name).schema
    payload_df = spark.createDataFrame([row], schema=schema)
    assignments = {column_name: f"source.{column_name}" for column_name in payload_df.columns}
    merge_single_row(
        workspace_id,
        hotel_guest_services_lakehouse_id,
        table_name,
        key_column,
        payload_df,
        assignments,
        assignments,
    )


def append_control_records(action_type: str, target_id: str, payload_json: str, status: str, pending_reconciliation: bool = False) -> None:
    confirmation_row = {
            "confirmation_id": confirmation_id,
            "action_type": action_type,
            "target_id": target_id,
            "status": status,
            "confirmed_at": datetime.strptime(request_timestamp_utc, "%Y-%m-%dT%H:%M:%SZ"),
            "idempotency_key": idempotency_key,
            "expected_record_version": expected_record_version,
            "correlation_id": None,
            "notebook_name": NOTEBOOK_NAME,
            "result_json": payload_json,
            "pending_reconciliation": pending_reconciliation,
    }
    merge_control_record("action_confirmations", "confirmation_id", confirmation_row)
    idempotency_row = {
            "idempotency_key": idempotency_key,
            "action_type": action_type,
            "result_id": target_id,
            "created_at": datetime.strptime(request_timestamp_utc, "%Y-%m-%dT%H:%M:%SZ"),
            "confirmation_id": confirmation_id,
            "request_hash": request_hash(payload_json),
            "status": status,
            "notebook_name": NOTEBOOK_NAME,
    }
    merge_control_record("idempotency_records", "idempotency_key", idempotency_row)
    audit_row = {
            "audit_event_id": f"AUD-{confirmation_id}",
            "entity_type": action_type,
            "entity_id": target_id,
            "action": action,
            "occurred_at": datetime.strptime(request_timestamp_utc, "%Y-%m-%dT%H:%M:%SZ"),
            "actor_type": "api",
            "confirmation_id": confirmation_id,
            "idempotency_key": idempotency_key,
            "payload_json": payload_json,
    }
    merge_control_record("audit_events", "audit_event_id", audit_row)


def verified_reservation_result(result_status: str, payload: dict[str, object]) -> dict[str, object]:
    persisted_rows = (
        read_delta(workspace_id, hotel_guest_services_lakehouse_id, "reservations")
        .where(F.col("reservation_id") == payload["reservation_id"])
        .select(
            "reservation_id", "guest_id", "property_id", "room_type_id", "check_in", "check_out",
            "status", "cancellation_policy", "room_count", "rate_total", "currency", "channel",
            "record_version", "last_confirmation_id", "pending_reconciliation",
        )
        .limit(2)
        .collect()
    )
    if len(persisted_rows) != 1:
        raise RuntimeError("Persisted reservation verification did not find exactly one row.")
    persisted = persisted_rows[0].asDict(recursive=True)
    expected = {
        "reservation_id": payload["reservation_id"],
        "guest_id": payload["guest_id"],
        "property_id": payload["property_id"],
        "room_type_id": payload["room_type_id"],
        "check_in": date.fromisoformat(str(payload["check_in"])),
        "check_out": date.fromisoformat(str(payload["check_out"])),
        "status": "cancelled" if action == "cancel" else str(payload.get("status", "future")),
        "cancellation_policy": payload["cancellation_policy"],
        "room_count": int(payload["room_count"]),
        "rate_total": Decimal(str(payload["rate_total"])),
        "currency": payload["currency"],
        "channel": payload["channel"],
        "last_confirmation_id": confirmation_id,
        "pending_reconciliation": False,
    }
    mismatches = [name for name, expected_value in expected.items() if persisted.get(name) != expected_value]
    if mismatches or int(persisted.get("record_version") or 0) < 1:
        raise RuntimeError(f"Persisted reservation verification failed for fields: {mismatches}")
    return {
        "status": result_status,
        "reservation_id": payload["reservation_id"],
        "confirmation_id": confirmation_id,
        "persisted_record": persisted,
    }

In [ ]:
NOTEBOOK_NAME = "HotelReservationAction"
require_evaluation_write_fence()
payload = load_json_payload(request_payload_json)
require_fields(payload, ["reservation_id", "guest_id", "property_id", "room_type_id", "check_in", "check_out", "room_count", "rate_total", "currency", "channel", "cancellation_policy"])
existing_idempotency = ensure_idempotency(workspace_id, hotel_guest_services_lakehouse_id, "reservation", request_payload_json)
if existing_idempotency is not None:
    if get_scalar(existing_idempotency, "result_id") != payload["reservation_id"]:
        raise RuntimeError("Conflict: idempotency record points to a different reservation.")
    result = verified_reservation_result("idempotent_replay", payload)
else:
    reservations = read_delta(workspace_id, hotel_guest_services_lakehouse_id, "reservations")
    guests = read_delta(workspace_id, hotel_guest_services_lakehouse_id, "guests")
    properties = read_delta(workspace_id, hotel_guest_services_lakehouse_id, "ops_properties")
    room_types = read_delta(workspace_id, hotel_guest_services_lakehouse_id, "ops_room_types")
    if guests.where(F.col("guest_id") == payload["guest_id"]).limit(1).count() != 1:
        raise ValueError("Guest does not exist.")
    property_row = properties.where(F.col("property_id") == payload["property_id"]).limit(1)
    if property_row.count() != 1:
        raise ValueError("Property does not exist.")
    room_type_row = room_types.where(F.col("room_type_id") == payload["room_type_id"]).limit(1)
    if room_type_row.count() != 1:
        raise ValueError("Room type does not exist.")
    if get_scalar(room_type_row, "property_id") != payload["property_id"]:
        raise ValueError("Room type does not belong to the requested property.")
    if get_scalar(property_row, "currency") != payload["currency"]:
        raise ValueError("Reservation currency must match the property currency.")
    check_in = date.fromisoformat(str(payload["check_in"]))
    check_out = date.fromisoformat(str(payload["check_out"]))
    if check_in < DATE_START or check_out > DATE_END or check_out <= check_in:
        raise ValueError("Reservation dates are outside the supported range.")
    existing = reservations.where(F.col("reservation_id") == payload["reservation_id"])
    existing_count = existing.limit(1).count()
    if action == "create" and existing_count and get_scalar(existing, "last_confirmation_id") == confirmation_id:
        append_control_records("reservation", str(payload["reservation_id"]), request_payload_json, "committed")
        result = verified_reservation_result("idempotent_replay", payload)
        notebookutils.notebook.exit(json.dumps(result, default=str))
    if action == "create" and existing_count:
        raise RuntimeError("Conflict: reservation already exists.")
    if action in {"change", "cancel"} and existing_count != 1:
        raise RuntimeError("Conflict: reservation does not exist.")
    if existing_count and expected_record_version is not None and int(get_scalar(existing, "record_version")) != expected_record_version:
        raise RuntimeError("Conflict: stale reservation record_version.")
    row_payload = {
        "reservation_id": payload["reservation_id"],
        "guest_id": payload["guest_id"],
        "property_id": payload["property_id"],
        "room_type_id": payload["room_type_id"],
        "check_in": check_in,
        "check_out": check_out,
        "status": "cancelled" if action == "cancel" else str(payload.get("status", "future")),
        "cancellation_policy": payload["cancellation_policy"],
        "room_count": int(payload["room_count"]),
        "rate_total": Decimal(str(payload["rate_total"])),
        "currency": payload["currency"],
        "channel": payload["channel"],
        "record_version": 1 if existing_count == 0 else int(get_scalar(existing, "record_version")) + 1,
        "last_modified_at": datetime.strptime(request_timestamp_utc, "%Y-%m-%dT%H:%M:%SZ"),
        "last_confirmation_id": confirmation_id,
        "pending_reconciliation": False,
        "is_fixture": False,
    }
    payload_df = spark.createDataFrame([row_payload])
    merge_single_row(
        workspace_id,
        hotel_guest_services_lakehouse_id,
        "reservations",
        "reservation_id",
        payload_df,
        {column_name: f"source.{column_name}" for column_name in payload_df.columns},
        {column_name: f"source.{column_name}" for column_name in payload_df.columns},
    )
    append_control_records("reservation", str(payload["reservation_id"]), request_payload_json, "committed")
    result = verified_reservation_result("committed", payload)

In [ ]:
notebookutils.notebook.exit(json.dumps(result, default=str))